# 🏆 DSN Bootcamp: DSN Mart Total Sales Prediction
**Objective:** Predict product sales across stores in Nigeria using 5-Fold Cross Validation, LightGBM, and XGBoost ensembling.  
**Evaluation Metric:** Root Mean Squared Error (RMSE)

## 1. Imports & Dynamic Data Loading (Kaggle Compatible)

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import OrdinalEncoder
import lightgbm as lgb
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Universal RMSE function (compatible with all scikit-learn versions)
def calculate_rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.array(y_true) - np.array(y_pred)) ** 2))

# Dynamic file finder (works locally and in Kaggle /kaggle/input)
def find_file(filename):
    # 1. Check current directory
    if os.path.exists(filename):
        return filename
    # 2. Check Kaggle input subdirectories
    kaggle_matches = glob.glob(f'/kaggle/input/**/{filename}', recursive=True)
    if kaggle_matches:
        return kaggle_matches[0]
    # 3. Check any recursive match
    matches = glob.glob(f'**/{filename}', recursive=True)
    if matches:
        return matches[0]
    raise FileNotFoundError(f"Could not locate {filename}. Please verify dataset is attached to the notebook.")

train_path = find_file('train.csv')
test_path = find_file('test.csv')
sub_path = find_file('sample_submission.csv')

print(f"Found train: {train_path}")
print(f"Found test : {test_path}")
print(f"Found sub  : {sub_path}")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_sub = pd.read_csv(sub_path)

print(f"\nTrain Shape: {train.shape}")
print(f"Test Shape : {test.shape}")
print(f"Sample Sub : {sample_sub.shape}")

## 2. Preprocessing & Advanced Feature Engineering

In [ ]:
train['is_train'] = 1
test['is_train'] = 0
df = pd.concat([train, test], ignore_index=True)

# 1. Clean Category Names
df['product_category'] = df['product_category'].astype(str).str.strip().str.title()

# 2. Impute Product Physical Weight by SKU
sku_weight = df.groupby('product_code')['product_weight_kg'].transform('mean')
cat_weight = df.groupby('product_category')['product_weight_kg'].transform('median')
df['product_weight_kg'] = df['product_weight_kg'].fillna(sku_weight).fillna(cat_weight)

# 3. Impute Shelf Visibility Zeros
df['shelf_visibility'] = df['shelf_visibility'].replace(0.0, np.nan)
sku_vis = df.groupby('product_code')['shelf_visibility'].transform('mean')
cat_vis = df.groupby('product_category')['shelf_visibility'].transform('median')
df['shelf_visibility'] = df['shelf_visibility'].fillna(sku_vis).fillna(cat_vis)

# 4. Clean Store Size & Non-Food Items
df['store_size'] = df['store_size'].fillna('Unknown')
non_food = ['Health And Hygiene', 'Household', 'Others']
df.loc[df['product_category'].isin(non_food), 'fat_content'] = 'Non-Edible'

# 5. Price & Elasticity Features
df['price_per_kg'] = df['product_price'] / df['product_weight_kg']
cat_p_mean = df.groupby('product_category')['product_price'].transform('mean')
cat_p_std = df.groupby('product_category')['product_price'].transform('std')
df['price_to_cat_mean'] = df['product_price'] / (cat_p_mean + 1e-5)
df['price_cat_zscore'] = (df['product_price'] - cat_p_mean) / (cat_p_std + 1e-5)

store_p_mean = df.groupby('store_code')['product_price'].transform('mean')
df['price_to_store_mean'] = df['product_price'] / (store_p_mean + 1e-5)

# 6. Shelf Visibility Promotion Features
sku_v_mean = df.groupby('product_code')['shelf_visibility'].transform('mean')
df['vis_to_sku_mean'] = df['shelf_visibility'] / (sku_v_mean + 1e-5)
cat_v_mean = df.groupby('product_category')['shelf_visibility'].transform('mean')
df['vis_to_cat_mean'] = df['shelf_visibility'] / (cat_v_mean + 1e-5)
store_v_mean = df.groupby('store_code')['shelf_visibility'].transform('mean')
df['vis_to_store_mean'] = df['shelf_visibility'] / (store_v_mean + 1e-5)

# 7. Assortment Breadth & Store Dynamics
df['product_store_count'] = df.groupby('product_code')['store_code'].transform('nunique')
df['store_product_count'] = df.groupby('store_code')['product_code'].transform('nunique')
df['is_corner_shop'] = (df['store_format'] == 'Corner Shop').astype(int)

df['store_format_tier'] = df['store_format'].astype(str) + '_' + df['store_location_tier'].astype(str)
df['store_format_size'] = df['store_format'].astype(str) + '_' + df['store_size'].astype(str)
df['store_cat_interaction'] = df['store_code'].astype(str) + '_' + df['product_category'].astype(str)
df['product_code_prefix'] = df['product_code'].astype(str).str.split('-').str[1].str[0]

def assign_price_tier(p):
    if p < 69: return 0
    elif p < 136: return 1
    elif p < 203: return 2
    else: return 3
df['price_tier'] = df['product_price'].apply(assign_price_tier)

# Split back to train and test
train_df = df[df['is_train'] == 1].reset_index(drop=True)
test_df = df[df['is_train'] == 0].reset_index(drop=True)

cat_cols = ['product_code', 'fat_content', 'product_category', 'store_code', 
            'store_size', 'store_location_tier', 'store_format', 'store_format_tier', 
            'store_format_size', 'store_cat_interaction', 'product_code_prefix']

feature_cols = [c for c in train_df.columns if c not in ['id', 'total_sales', 'is_train']]
print(f"Total Engineered Features: {len(feature_cols)}")

## 3. 5-Fold Cross-Validation & Model Training

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
X_train_raw = train_df[feature_cols].copy()
y_train = train_df['total_sales'].values
X_test_raw = test_df[feature_cols].copy()

lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'n_estimators': 1500,
    'learning_rate': 0.0288,
    'num_leaves': 28,
    'max_depth': 3,
    'min_child_samples': 52,
    'subsample': 0.8978,
    'colsample_bytree': 0.5447,
    'reg_alpha': 0.7570,
    'reg_lambda': 0.00267,
    'random_state': 42,
    'verbose': -1,
    'n_jobs': 4
}

xgb_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'n_estimators': 1500,
    'learning_rate': 0.0414,
    'max_depth': 4,
    'min_child_weight': 3,
    'subsample': 0.9384,
    'colsample_bytree': 0.5120,
    'reg_alpha': 0.0333,
    'reg_lambda': 3.4488,
    'random_state': 42,
    'tree_method': 'hist',
    'n_jobs': 4
}

test_preds_lgb = np.zeros(len(test_df))
test_preds_xgb = np.zeros(len(test_df))
oof_lgb = np.zeros(len(train_df))
oof_xgb = np.zeros(len(train_df))

print("--- Starting 5-Fold Training ---")
for fold, (trn_idx, val_idx) in enumerate(kf.split(X_train_raw, y_train)):
    X_trn = X_train_raw.iloc[trn_idx].copy()
    y_trn = y_train[trn_idx]
    X_val = X_train_raw.iloc[val_idx].copy()
    y_val = y_train[val_idx]
    X_tst = X_test_raw.copy()
    
    # Out-of-fold target encodings strictly on train split
    tmp = pd.DataFrame({'store': X_trn['store_code'], 'cat': X_trn['product_category'], 'store_cat': X_trn['store_cat_interaction'], 'target': y_trn})
    sm = tmp.groupby('store', observed=False)['target'].mean()
    cm = tmp.groupby('cat', observed=False)['target'].mean()
    scm = tmp.groupby('store_cat', observed=False)['target'].mean()
    ov = y_trn.mean()
    
    for d in [X_trn, X_val, X_tst]:
        d['te_store'] = d['store_code'].map(sm).fillna(ov).astype(float)
        d['te_cat'] = d['product_category'].map(cm).fillna(ov).astype(float)
        d['te_store_cat'] = d['store_cat_interaction'].map(scm).fillna(ov).astype(float)
    
    # LightGBM (native categorical support)
    X_trn_lgb, X_val_lgb, X_tst_lgb = X_trn.copy(), X_val.copy(), X_tst.copy()
    for col in cat_cols:
        X_trn_lgb[col] = X_trn_lgb[col].astype('category')
        X_val_lgb[col] = X_val_lgb[col].astype('category')
        X_tst_lgb[col] = X_tst_lgb[col].astype('category')
    
    lgb_m = lgb.LGBMRegressor(**lgb_params)
    lgb_m.fit(X_trn_lgb, y_trn, eval_set=[(X_val_lgb, y_val)], callbacks=[lgb.early_stopping(60, verbose=False)])
    oof_lgb[val_idx] = lgb_m.predict(X_val_lgb)
    test_preds_lgb += lgb_m.predict(X_tst_lgb) / kf.n_splits
    
    # XGBoost (Ordinal encoded to handle unseen test categories seamlessly)
    X_trn_xgb, X_val_xgb, X_tst_xgb = X_trn.copy(), X_val.copy(), X_tst.copy()
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_trn_xgb[cat_cols] = oe.fit_transform(X_trn_xgb[cat_cols].astype(str))
    X_val_xgb[cat_cols] = oe.transform(X_val_xgb[cat_cols].astype(str))
    X_tst_xgb[cat_cols] = oe.transform(X_tst_xgb[cat_cols].astype(str))
    
    xgb_m = xgb.XGBRegressor(**xgb_params)
    xgb_m.fit(X_trn_xgb, y_trn, eval_set=[(X_val_xgb, y_val)], verbose=False)
    oof_xgb[val_idx] = xgb_m.predict(X_val_xgb)
    test_preds_xgb += xgb_m.predict(X_tst_xgb) / kf.n_splits
    
    print(f"Fold {fold+1} complete | LightGBM RMSE: {calculate_rmse(y_val, oof_lgb[val_idx]):.4f} | XGBoost RMSE: {calculate_rmse(y_val, oof_xgb[val_idx]):.4f}")

print("\n--- Overall Cross-Validation Scores ---")
print(f"LightGBM OOF RMSE : {calculate_rmse(y_train, oof_lgb):.4f}")
print(f"XGBoost OOF RMSE  : {calculate_rmse(y_train, oof_xgb):.4f}")

w_lgb, w_xgb = 0.898, 0.102
oof_blend = w_lgb * oof_lgb + w_xgb * oof_xgb
print(f"Final 5-Fold Ensemble OOF RMSE: {calculate_rmse(y_train, oof_blend):.4f}")

## 4. Generate & Save Submission File

In [ ]:
final_test_preds = w_lgb * test_preds_lgb + w_xgb * test_preds_xgb
final_test_preds = np.clip(final_test_preds, a_min=float(y_train.min()), a_max=None)

submission = pd.DataFrame({
    'id': test_df['id'],
    'total_sales': np.round(final_test_preds, 4)
})

# Sanity assertions
assert submission.shape == sample_sub.shape, f"Shape mismatch: {submission.shape} vs {sample_sub.shape}"
assert (submission['id'] == sample_sub['id']).all(), "ID ordering mismatch!"
assert submission['total_sales'].isnull().sum() == 0, "Contains NaNs!"
assert (submission['total_sales'] < 0).sum() == 0, "Contains negative predictions!"

# Write output submission
submission.to_csv('submission.csv', index=False)
print("\n>>> submission.csv created successfully!")
print(f"Total prediction rows: {len(submission)}")
submission.head(10)